In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
actual = pd.read_csv(f"{path}/actual_portfolio_load.csv")
load = pd.read_csv(f"{path}/forecast_load.csv")
imb = pd.read_csv(f"{path}/imbalance_prices.csv")
da = pd.read_csv(f"{path}/day_ahead_prices.csv")

In [3]:
grid = actual[["timestamp_utc", "timestamp_local", "total_actual_mwh"]].merge(
    load[["timestamp_utc", "total_forecast_mwh"]], on="timestamp_utc").merge(
    imb[["timestamp_utc", "imbalance_price_eur_mwh"]], on="timestamp_utc").merge(
    da[["timestamp_utc", "day_ahead_price_eur_mwh"]], on="timestamp_utc")
A = grid["total_actual_mwh"].to_numpy()
P_imb = grid["imbalance_price_eur_mwh"].to_numpy()
P_da = grid["day_ahead_price_eur_mwh"].to_numpy()

In [4]:
grid["month"] = grid["timestamp_local"].str[5:7]

In [5]:
imbalance_volume = A - grid["total_forecast_mwh"].to_numpy()
imbalance_cost = imbalance_volume * P_imb  #contains the penalty
premium_imb = imbalance_volume @ (P_imb - P_da)

In [6]:
rows = []
for name, tag in [("UNHEDGED", "None"), ("COARSE", "coarse"), ("GRANULAR", "granular")]:
    residual = pd.read_csv(f"{path}/residual_" + tag + ".csv")["day_ahead_residual"].to_numpy()
    if tag == "None":
        futures_cost = np.zeros(len(grid))
    else:
        futures_cost = pd.read_csv(f"{path}/task3_hedge_" + tag + ".csv")["futures_cost"].to_numpy()

    # Total cost of each quarter-hour: futures, Day-Ahead residual, imbalance
    cost = futures_cost + residual * P_da + imbalance_cost

    # Average procurement price per month, then its spread across the twelve months
    monthly = pd.DataFrame({"month": grid["month"], "cost": cost, "volume": A}).groupby("month").sum()
    monthly_price = monthly["cost"] / monthly["volume"]
    rows.append({
        "strategy": name,
        "final_cost_eur": cost.sum(),
        "average_price_eur_mwh": cost.sum() / A.sum(),
        "saving_vs_unhedged_eur": rows[0]["final_cost_eur"] - cost.sum() if rows else 0.0,
        "monthly_price_std_eur_mwh": monthly_price.std(),
        "monthly_price_range_eur_mwh": monthly_price.max() - monthly_price.min(),
        "abs_residual_da_volume_mwh": np.abs(residual).sum(),
        "residual_rmse_mwh": np.sqrt(np.mean(residual ** 2)),
        "imbalance_cost_eur": imbalance_cost.sum(),
        "imbalance_premium_eur": premium_imb,
    })

In [7]:
table = pd.DataFrame(rows).set_index("strategy").round(2)
print(table.T.to_string())
table.to_csv(f"{path}/task6_strategy_comparison.csv")

strategy                       UNHEDGED      COARSE    GRANULAR
final_cost_eur               3537207.83  3481732.71  3482217.13
average_price_eur_mwh             82.00       80.71       80.72
saving_vs_unhedged_eur             0.00    55475.12    54990.70
monthly_price_std_eur_mwh         20.15        2.49       18.56
monthly_price_range_eur_mwh       58.43        9.20       47.17
abs_residual_da_volume_mwh     43000.00    12730.37    11973.73
residual_rmse_mwh                  1.32        0.44        0.42
imbalance_cost_eur             30503.84    30503.84    30503.84
imbalance_premium_eur          13589.24    13589.24    13589.24
